In [1]:
# ============================================================
# 1. Install & Imports
# ============================================================

!pip install -q -U ultralytics

import os
import json
import random
import shutil
import yaml
import xml.etree.ElementTree as ET

from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO
from google.colab import drive

drive.mount("/content/drive")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Mounted at /content/drive


In [2]:
# ============================================================
# 2. Path Setting
# ============================================================

# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

DRIVE_BASE = Path(
    "/content/drive/MyDrive/SAR_AI_Ship_Detection"
)

HRSID_DRIVE = DRIVE_BASE / "HRSID"
LSSSDD_DRIVE = DRIVE_BASE / "LS-SSDD"

HRSID_MODEL_PATH = (
    DRIVE_BASE
    / "trained_models"
    / "hrsid_yolo26s_20epoch_best.pt"
)

# ------------------------------------------------------------
# Local runtime
# ------------------------------------------------------------

HRSID_LOCAL = Path("/content/HRSID")
LSSSDD_LOCAL = Path("/content/LS-SSDD")

HRSID_YOLO = Path("/content/HRSID-YOLO")
LSSSDD_YOLO = Path("/content/LS-SSDD-YOLO")

MIXED_DIR = Path("/content/HRSID_LSSDD_5to1")

print("HRSID:", HRSID_DRIVE)
print("LS-SSDD:", LSSSDD_DRIVE)
print("Model:", HRSID_MODEL_PATH)

HRSID: /content/drive/MyDrive/SAR_AI_Ship_Detection/HRSID
LS-SSDD: /content/drive/MyDrive/SAR_AI_Ship_Detection/LS-SSDD
Model: /content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models/hrsid_yolo26s_20epoch_best.pt


In [3]:
# ============================================================
# 3. Copy datasets to local runtime
# ============================================================

if not HRSID_LOCAL.exists():
    shutil.copytree(HRSID_DRIVE, HRSID_LOCAL)
    print("HRSID copied to /content")
else:
    print("HRSID already exists")

if not LSSSDD_LOCAL.exists():
    shutil.copytree(LSSSDD_DRIVE, LSSSDD_LOCAL)
    print("LS-SSDD copied to /content")
else:
    print("LS-SSDD already exists")

HRSID copied to /content
LS-SSDD copied to /content


In [4]:
# ============================================================
# 4. HRSID split
# ============================================================

HRSID_IMAGE_DIR = HRSID_LOCAL / "images"
HRSID_ANN_DIR = HRSID_LOCAL / "annotations"

HRSID_TRAIN_JSON = HRSID_ANN_DIR / "train2017.json"
HRSID_TEST_JSON = HRSID_ANN_DIR / "test2017.json"


with open(HRSID_TRAIN_JSON, "r") as f:
    hrsid_train_coco = json.load(f)

with open(HRSID_TEST_JSON, "r") as f:
    hrsid_test_coco = json.load(f)


# ------------------------------------------------------------
# Official train IDs
# ------------------------------------------------------------

hrsid_all_train_ids = [
    img["id"]
    for img in hrsid_train_coco["images"]
]

random.seed(42)
random.shuffle(hrsid_all_train_ids)

n_val = int(len(hrsid_all_train_ids) * 0.10)

hrsid_val_ids = hrsid_all_train_ids[:n_val]
hrsid_train_ids = hrsid_all_train_ids[n_val:]

hrsid_test_ids = [
    img["id"]
    for img in hrsid_test_coco["images"]
]


print("HRSID train:", len(hrsid_train_ids))
print("HRSID val  :", len(hrsid_val_ids))
print("HRSID test :", len(hrsid_test_ids))

HRSID train: 3278
HRSID val  : 364
HRSID test : 1962


In [5]:
# ============================================================
# 5. Index HRSID images
# ============================================================

image_extensions = {
    ".png", ".jpg", ".jpeg",
    ".tif", ".tiff"
}

hrsid_image_index = {}

for path in HRSID_IMAGE_DIR.rglob("*"):

    if path.is_file() and path.suffix.lower() in image_extensions:

        hrsid_image_index[path.name] = path


print("Indexed HRSID images:", len(hrsid_image_index))

Indexed HRSID images: 5606


In [6]:
# ============================================================
# 6. Convert HRSID COCO → YOLO
# ============================================================

if HRSID_YOLO.exists():
    shutil.rmtree(HRSID_YOLO)


def build_coco_index(coco):

    image_info = {
        img["id"]: img
        for img in coco["images"]
    }

    annotations = defaultdict(list)

    for ann in coco["annotations"]:
        annotations[ann["image_id"]].append(ann)

    return image_info, annotations


train_image_info, train_annotations = build_coco_index(
    hrsid_train_coco
)

test_image_info, test_annotations = build_coco_index(
    hrsid_test_coco
)


def convert_hrsid_split(
    image_ids,
    split,
    image_info,
    annotations
):

    img_out = HRSID_YOLO / "images" / split
    lbl_out = HRSID_YOLO / "labels" / split

    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    annotation_count = 0
    missing_count = 0

    for image_id in image_ids:

        info = image_info[image_id]

        filename = info["file_name"]

        src_image = hrsid_image_index.get(
            Path(filename).name
        )

        if src_image is None:

            missing_count += 1
            continue

        width = info["width"]
        height = info["height"]

        dst_image = img_out / src_image.name

        shutil.copy2(src_image, dst_image)

        label_path = (
            lbl_out /
            f"{src_image.stem}.txt"
        )

        lines = []

        for ann in annotations.get(image_id, []):

            x, y, w, h = ann["bbox"]

            xc = (x + w / 2) / width
            yc = (y + h / 2) / height
            wn = w / width
            hn = h / height

            lines.append(
                f"0 "
                f"{xc:.6f} "
                f"{yc:.6f} "
                f"{wn:.6f} "
                f"{hn:.6f}"
            )

            annotation_count += 1

        with open(label_path, "w") as f:
            f.write("\n".join(lines))

    print(
        split,
        "| images:",
        len(list(img_out.iterdir())),
        "| labels:",
        len(list(lbl_out.iterdir())),
        "| annotations:",
        annotation_count,
        "| missing:",
        missing_count
    )


convert_hrsid_split(
    hrsid_train_ids,
    "train",
    train_image_info,
    train_annotations
)

convert_hrsid_split(
    hrsid_val_ids,
    "val",
    train_image_info,
    train_annotations
)

convert_hrsid_split(
    hrsid_test_ids,
    "test",
    test_image_info,
    test_annotations
)

train | images: 3278 | labels: 3278 | annotations: 9882 | missing: 0
val | images: 364 | labels: 364 | annotations: 1165 | missing: 0
test | images: 1962 | labels: 1962 | annotations: 5922 | missing: 0


In [7]:
# ============================================================
# 7. LS-SSDD split
# ============================================================

LSSSDD_IMG_TRAIN = (
    LSSSDD_LOCAL /
    "JPEGImages_sub_train"
)

LSSSDD_IMG_TEST = (
    LSSSDD_LOCAL /
    "JPEGImages_sub_test"
)

LSSSDD_ANN_DIR = (
    LSSSDD_LOCAL /
    "Annotations_sub"
)

TRAIN_TXT = LSSSDD_LOCAL / "train.txt"
VAL_TXT = LSSSDD_LOCAL / "val.txt"
TEST_TXT = LSSSDD_LOCAL / "test.txt"


def read_ids(txt_path):

    with open(txt_path, "r") as f:

        return [
            line.strip()
            for line in f
            if line.strip()
        ]


all_train_ids = read_ids(TRAIN_TXT)
lsssdd_val_ids = read_ids(VAL_TXT)
lsssdd_test_ids = read_ids(TEST_TXT)


# ------------------------------------------------------------
# Remove validation IDs from original train
# ------------------------------------------------------------

val_set = set(lsssdd_val_ids)

lsssdd_train_ids = [
    image_id
    for image_id in all_train_ids
    if image_id not in val_set
]


print(
    "LS-SSDD train:",
    len(lsssdd_train_ids)
)

print(
    "LS-SSDD val:",
    len(lsssdd_val_ids)
)

print(
    "LS-SSDD test:",
    len(lsssdd_test_ids)
)


print(
    "train ∩ val:",
    len(
        set(lsssdd_train_ids)
        &
        set(lsssdd_val_ids)
    )
)

print(
    "train ∩ test:",
    len(
        set(lsssdd_train_ids)
        &
        set(lsssdd_test_ids)
    )
)

LS-SSDD train: 5100
LS-SSDD val: 900
LS-SSDD test: 3000
train ∩ val: 0
train ∩ test: 0


In [8]:
# ============================================================
# 8. LS-SSDD VOC → YOLO
# ============================================================

if LSSSDD_YOLO.exists():
    shutil.rmtree(LSSSDD_YOLO)


def convert_voc_to_yolo(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")

    width = float(size.find("width").text)
    height = float(size.find("height").text)

    labels = []

    for obj in root.findall("object"):

        bbox = obj.find("bndbox")

        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        xmin = max(0, min(xmin, width))
        xmax = max(0, min(xmax, width))
        ymin = max(0, min(ymin, height))
        ymax = max(0, min(ymax, height))

        bw = xmax - xmin
        bh = ymax - ymin

        if bw <= 0 or bh <= 0:
            continue

        xc = (xmin + xmax) / 2 / width
        yc = (ymin + ymax) / 2 / height
        bw /= width
        bh /= height

        labels.append(
            f"0 "
            f"{xc:.6f} "
            f"{yc:.6f} "
            f"{bw:.6f} "
            f"{bh:.6f}"
        )

    return labels

In [9]:
# ============================================================
# 9. Prepare LS-SSDD splits
# ============================================================

def find_image(image_id, image_dir):

    candidates = [
        image_dir / f"{image_id}.jpg",
        image_dir / f"{image_id}.jpeg",
        image_dir / f"{image_id}.png",
        image_dir / f"{image_id}.tif"
    ]

    for p in candidates:

        if p.exists():
            return p

    return None


def prepare_lsssdd_split(
    ids,
    split,
    image_dir
):

    img_out = (
        LSSSDD_YOLO /
        "images" /
        split
    )

    lbl_out = (
        LSSSDD_YOLO /
        "labels" /
        split
    )

    img_out.mkdir(
        parents=True,
        exist_ok=True
    )

    lbl_out.mkdir(
        parents=True,
        exist_ok=True
    )

    missing = 0
    instance_count = 0

    for image_id in ids:

        src_img = find_image(
            image_id,
            image_dir
        )

        if src_img is None:

            missing += 1
            continue

        dst_img = (
            img_out /
            src_img.name
        )

        shutil.copy2(
            src_img,
            dst_img
        )

        xml_path = (
            LSSSDD_ANN_DIR /
            f"{image_id}.xml"
        )

        label_path = (
            lbl_out /
            f"{src_img.stem}.txt"
        )

        labels = []

        if xml_path.exists():

            labels = convert_voc_to_yolo(
                xml_path
            )

        with open(label_path, "w") as f:

            f.write(
                "\n".join(labels)
            )

        instance_count += len(labels)

    print(
        split,
        "| images:",
        len(list(img_out.iterdir())),
        "| labels:",
        len(list(lbl_out.iterdir())),
        "| instances:",
        instance_count,
        "| missing:",
        missing
    )


prepare_lsssdd_split(
    lsssdd_train_ids,
    "train",
    LSSSDD_IMG_TRAIN
)

prepare_lsssdd_split(
    lsssdd_val_ids,
    "val",
    LSSSDD_IMG_TRAIN
)

prepare_lsssdd_split(
    lsssdd_test_ids,
    "test",
    LSSSDD_IMG_TEST
)

train | images: 5100 | labels: 5100 | instances: 2938 | missing: 0
val | images: 900 | labels: 900 | instances: 699 | missing: 0
test | images: 3000 | labels: 3000 | instances: 2378 | missing: 0


In [10]:
# ============================================================
# 10. Create Full Mixed Pool
#     HRSID train 3278 + LS-SSDD train 5100
# ============================================================

MIXED_DIR = Path("/content/HRSID_LSSDD_WEIGHTED")

if MIXED_DIR.exists():
    shutil.rmtree(MIXED_DIR)


MIX_TRAIN_IMG = MIXED_DIR / "images/train"
MIX_TRAIN_LBL = MIXED_DIR / "labels/train"

MIX_VAL_IMG = MIXED_DIR / "images/val"
MIX_VAL_LBL = MIXED_DIR / "labels/val"

MIX_TRAIN_IMG.mkdir(parents=True, exist_ok=True)
MIX_TRAIN_LBL.mkdir(parents=True, exist_ok=True)

MIX_VAL_IMG.mkdir(parents=True, exist_ok=True)
MIX_VAL_LBL.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Source folders
# ------------------------------------------------------------

HRSID_TRAIN_IMG = HRSID_YOLO / "images/train"
HRSID_TRAIN_LBL = HRSID_YOLO / "labels/train"

LSSSDD_TRAIN_IMG = LSSSDD_YOLO / "images/train"
LSSSDD_TRAIN_LBL = LSSSDD_YOLO / "labels/train"


hrsid_train_images = sorted(HRSID_TRAIN_IMG.iterdir())
lsssdd_train_images = sorted(LSSSDD_TRAIN_IMG.iterdir())


def add_dataset(images, label_dir, prefix):

    for img_path in images:

        new_img_name = f"{prefix}_{img_path.name}"

        dst_img = MIX_TRAIN_IMG / new_img_name

        # local runtime이라 symlink 사용
        os.symlink(img_path, dst_img)

        src_label = label_dir / f"{img_path.stem}.txt"

        dst_label = (
            MIX_TRAIN_LBL /
            f"{Path(new_img_name).stem}.txt"
        )

        if src_label.exists():
            shutil.copy2(src_label, dst_label)
        else:
            dst_label.touch()


# HRSID 전체
add_dataset(
    hrsid_train_images,
    HRSID_TRAIN_LBL,
    "hrsid"
)

# LS-SSDD 전체
add_dataset(
    lsssdd_train_images,
    LSSSDD_TRAIN_LBL,
    "lsssdd"
)


print("HRSID pool   :", len(hrsid_train_images))
print("LS-SSDD pool :", len(lsssdd_train_images))
print(
    "Total pool   :",
    len(list(MIX_TRAIN_IMG.iterdir()))
)

HRSID pool   : 3278
LS-SSDD pool : 5100
Total pool   : 8378


In [11]:
# ============================================================
# 11. HRSID Validation Set
# ============================================================

for img_path in (HRSID_YOLO / "images/val").iterdir():

    dst_img = MIX_VAL_IMG / img_path.name

    os.symlink(
        img_path,
        dst_img
    )

    src_label = (
        HRSID_YOLO /
        "labels/val" /
        f"{img_path.stem}.txt"
    )

    dst_label = (
        MIX_VAL_LBL /
        f"{img_path.stem}.txt"
    )

    shutil.copy2(
        src_label,
        dst_label
    )


print(
    "Validation images:",
    len(list(MIX_VAL_IMG.iterdir()))
)

Validation images: 364


In [12]:
# ============================================================
# 12. Mixed Dataset YAML
# ============================================================

MIXED_YAML = MIXED_DIR / "data.yaml"

mixed_yaml = {
    "path": str(MIXED_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "ship"
    }
}


with open(MIXED_YAML, "w") as f:

    yaml.safe_dump(
        mixed_yaml,
        f,
        sort_keys=False
    )


print(MIXED_YAML.read_text())

path: /content/HRSID_LSSDD_WEIGHTED
train: images/train
val: images/val
names:
  0: ship



In [13]:

# ============================================================
# 13. Dynamic Mixed Dataset Setting
#
# HRSID : LS-SSDD = RATIO : 1
#
# - HRSID train: ALL images every epoch
# - LS-SSDD train: random subset every epoch
# - HRSID val: fixed validation set
# ============================================================

import os
import random
import shutil
import yaml
from pathlib import Path


# ============================================================
# 1. Ratio Setting
# ============================================================

# 3:1 -> RATIO = 3
# 2:1 -> RATIO = 2

RATIO = 3


# ============================================================
# 2. Source Dataset Paths
# ============================================================

HRSID_TRAIN_IMG = HRSID_YOLO / "images/train"
HRSID_TRAIN_LBL = HRSID_YOLO / "labels/train"

HRSID_VAL_IMG = HRSID_YOLO / "images/val"
HRSID_VAL_LBL = HRSID_YOLO / "labels/val"

LSSSDD_TRAIN_IMG = LSSSDD_YOLO / "images/train"
LSSSDD_TRAIN_LBL = LSSSDD_YOLO / "labels/train"


# ============================================================
# 3. Dynamic Mixed Dataset Paths
# ============================================================

MIXED_DIR = Path("/content/HRSID_LSSDD_DYNAMIC")

MIX_TRAIN_IMG = MIXED_DIR / "images/train"
MIX_TRAIN_LBL = MIXED_DIR / "labels/train"

MIX_VAL_IMG = MIXED_DIR / "images/val"
MIX_VAL_LBL = MIXED_DIR / "labels/val"

MIXED_YAML = MIXED_DIR / "data.yaml"


# ============================================================
# 4. Check Source Dataset
# ============================================================

assert HRSID_TRAIN_IMG.exists(), (
    f"HRSID train image folder not found:\n"
    f"{HRSID_TRAIN_IMG}"
)

assert HRSID_TRAIN_LBL.exists(), (
    f"HRSID train label folder not found:\n"
    f"{HRSID_TRAIN_LBL}"
)

assert HRSID_VAL_IMG.exists(), (
    f"HRSID val image folder not found:\n"
    f"{HRSID_VAL_IMG}"
)

assert HRSID_VAL_LBL.exists(), (
    f"HRSID val label folder not found:\n"
    f"{HRSID_VAL_LBL}"
)

assert LSSSDD_TRAIN_IMG.exists(), (
    f"LS-SSDD train image folder not found:\n"
    f"{LSSSDD_TRAIN_IMG}"
)

assert LSSSDD_TRAIN_LBL.exists(), (
    f"LS-SSDD train label folder not found:\n"
    f"{LSSSDD_TRAIN_LBL}"
)


# ============================================================
# 5. Index Training Images
# ============================================================

hrsid_images = sorted(
    [
        p
        for p in HRSID_TRAIN_IMG.iterdir()
        if p.is_file()
    ]
)

lsssdd_images = sorted(
    [
        p
        for p in LSSSDD_TRAIN_IMG.iterdir()
        if p.is_file()
    ]
)


N_HRSID = len(hrsid_images)
N_LSSDD_TOTAL = len(lsssdd_images)


# LS subset size per epoch
N_LSSDD_PER_EPOCH = round(
    N_HRSID / RATIO
)


assert N_LSSDD_PER_EPOCH <= N_LSSDD_TOTAL, (
    "Requested LS-SSDD subset is larger than "
    "the available LS-SSDD training set."
)


print("=" * 60)
print("Dynamic Mixed Training Setting")
print("=" * 60)

print(
    "Ratio               :",
    f"HRSID : LS-SSDD = {RATIO} : 1"
)

print(
    "HRSID train total   :",
    N_HRSID
)

print(
    "LS-SSDD train total :",
    N_LSSDD_TOTAL
)

print(
    "LS-SSDD / epoch     :",
    N_LSSDD_PER_EPOCH
)

print(
    "Total / epoch       :",
    N_HRSID + N_LSSDD_PER_EPOCH
)

print("=" * 60)


# ============================================================
# 6. Reset / Prepare HRSID Validation Dataset
#
# Validation set is fixed.
# It is NOT rebuilt every epoch.
# ============================================================

if MIX_VAL_IMG.exists():
    shutil.rmtree(
        MIX_VAL_IMG
    )

if MIX_VAL_LBL.exists():
    shutil.rmtree(
        MIX_VAL_LBL
    )


MIX_VAL_IMG.mkdir(
    parents=True,
    exist_ok=True
)

MIX_VAL_LBL.mkdir(
    parents=True,
    exist_ok=True
)


hrsid_val_images = sorted(
    [
        p
        for p in HRSID_VAL_IMG.iterdir()
        if p.is_file()
    ]
)


for img_path in hrsid_val_images:

    # --------------------------------------------------------
    # Image -> symbolic link
    # --------------------------------------------------------

    dst_img = (
        MIX_VAL_IMG
        / img_path.name
    )

    if dst_img.exists() or dst_img.is_symlink():
        dst_img.unlink()

    os.symlink(
        img_path,
        dst_img
    )


    # --------------------------------------------------------
    # Label -> copy
    # --------------------------------------------------------

    src_label = (
        HRSID_VAL_LBL
        / f"{img_path.stem}.txt"
    )

    dst_label = (
        MIX_VAL_LBL
        / f"{img_path.stem}.txt"
    )


    assert src_label.exists(), (
        f"Missing HRSID validation label:\n"
        f"{src_label}"
    )


    shutil.copy2(
        src_label,
        dst_label
    )


# ============================================================
# 7. Check Validation Dataset
# ============================================================

n_val_images = len(
    list(
        MIX_VAL_IMG.iterdir()
    )
)

n_val_labels = len(
    list(
        MIX_VAL_LBL.glob("*.txt")
    )
)


print()
print("=" * 60)
print("Dynamic HRSID Validation Dataset")
print("=" * 60)

print(
    "Images :",
    n_val_images
)

print(
    "Labels :",
    n_val_labels
)

print("=" * 60)


assert n_val_images == len(hrsid_val_images)

assert n_val_images == n_val_labels


# Expected for current HRSID split
assert n_val_images == 364, (
    f"Unexpected HRSID validation size: "
    f"{n_val_images}"
)


# ============================================================
# 8. Create Dynamic data.yaml
# ============================================================

MIXED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


mixed_yaml = {

    "path": str(
        MIXED_DIR
    ),

    "train": "images/train",

    "val": "images/val",

    "names": {
        0: "ship"
    }
}


with open(
    MIXED_YAML,
    "w"
) as f:

    yaml.safe_dump(
        mixed_yaml,
        f,
        sort_keys=False
    )


# ============================================================
# 9. Check Dynamic YAML
# ============================================================

print()
print("=" * 60)
print("Dynamic data.yaml")
print("=" * 60)

print(
    MIXED_YAML.read_text()
)

print("=" * 60)


assert MIXED_YAML.exists()

assert str(MIXED_DIR) in (
    MIXED_YAML.read_text()
)


print()
print(
    "Cell 13 setup complete."
)

Dynamic Mixed Training Setting
Ratio               : HRSID : LS-SSDD = 3 : 1
HRSID train total   : 3278
LS-SSDD train total : 5100
LS-SSDD / epoch     : 1093
Total / epoch       : 4371

Dynamic HRSID Validation Dataset
Images : 364
Labels : 364

Dynamic data.yaml
path: /content/HRSID_LSSDD_DYNAMIC
train: images/train
val: images/val
names:
  0: ship


Cell 13 setup complete.


In [14]:
# ============================================================
# 14. Function to create mixed dataset
# ============================================================

def rebuild_mixed_train(epoch):

    # --------------------------------------------------------
    # remove previous train set
    # --------------------------------------------------------

    if MIX_TRAIN_IMG.exists():
        shutil.rmtree(MIX_TRAIN_IMG)

    if MIX_TRAIN_LBL.exists():
        shutil.rmtree(MIX_TRAIN_LBL)


    MIX_TRAIN_IMG.mkdir(
        parents=True,
        exist_ok=True
    )

    MIX_TRAIN_LBL.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # 1. Add ALL HRSID
    # --------------------------------------------------------

    for img_path in hrsid_images:

        new_name = (
            "hrsid_" + img_path.name
        )

        dst_img = (
            MIX_TRAIN_IMG /
            new_name
        )

        os.symlink(
            img_path,
            dst_img
        )


        src_label = (
            HRSID_TRAIN_LBL /
            f"{img_path.stem}.txt"
        )

        dst_label = (
            MIX_TRAIN_LBL /
            f"{Path(new_name).stem}.txt"
        )

        shutil.copy2(
            src_label,
            dst_label
        )


    # --------------------------------------------------------
    # 2. Random LS-SSDD subset
    # --------------------------------------------------------

    rng = random.Random(
        42 + epoch
    )

    selected_ls = rng.sample(
        lsssdd_images,
        N_LSSDD_PER_EPOCH
    )


    for img_path in selected_ls:

        new_name = (
            "lsssdd_" + img_path.name
        )

        dst_img = (
            MIX_TRAIN_IMG /
            new_name
        )

        os.symlink(
            img_path,
            dst_img
        )


        src_label = (
            LSSSDD_TRAIN_LBL /
            f"{img_path.stem}.txt"
        )

        dst_label = (
            MIX_TRAIN_LBL /
            f"{Path(new_name).stem}.txt"
        )

        shutil.copy2(
            src_label,
            dst_label
        )


    print()
    print(
        f"Epoch {epoch + 1}"
    )

    print(
        "HRSID   :",
        len(hrsid_images)
    )

    print(
        "LS-SSDD :",
        len(selected_ls)
    )

    print(
        "Total   :",
        len(
            list(
                MIX_TRAIN_IMG.iterdir()
            )
        )
    )


    return selected_ls

In [15]:
# ============================================================
# 15. Dynamic 2:1 Fine-Tuning + Global Best Tracking
# ============================================================

from ultralytics import YOLO
from pathlib import Path

TOTAL_EPOCHS = 5

current_model_path = HRSID_MODEL_PATH

global_best_map50 = -1.0
global_best_stage = -1
global_best_path = None


for epoch in range(TOTAL_EPOCHS):

    stage = epoch + 1

    print()
    print("=" * 60)
    print(f"Fine-tuning stage {stage}/{TOTAL_EPOCHS}")
    print("=" * 60)


    # --------------------------------------------------------
    # HRSID 전체 + 새로운 LS-SSDD subset 생성
    # --------------------------------------------------------

    rebuild_mixed_train(epoch)


    # --------------------------------------------------------
    # 이전 stage의 last.pt에서 시작
    # --------------------------------------------------------

    model = YOLO(
        str(current_model_path)
    )


    run_name = (
        f"dynamic3to1_5epoch_"
        f"stage_{stage}"
    )


    # --------------------------------------------------------
    # 1 epoch 학습
    # --------------------------------------------------------

    results = model.train(
        data=str(MIXED_YAML),
        epochs=1,
        imgsz=800,
        batch=16,
        device=0,
        workers=2,
        optimizer="AdamW",
        lr0=1e-5,
        project="/content/runs",
        name=run_name
    )


    # --------------------------------------------------------
    # 현재 stage 결과 경로
    # --------------------------------------------------------

    # 실제 Ultralytics가 생성한 run directory 사용
    RUN_DIR = Path(model.trainer.save_dir)

    STAGE_BEST = (
        RUN_DIR
        / "weights"
        / "best.pt"
    )

    STAGE_LAST = (
        RUN_DIR
        / "weights"
        / "last.pt"
    )


    assert STAGE_BEST.exists()
    assert STAGE_LAST.exists()


    # --------------------------------------------------------
    # Validation mAP50
    # --------------------------------------------------------

    current_map50 = float(
        results.results_dict[
            "metrics/mAP50(B)"
        ]
    )


    print(
        f"Stage {stage} "
        f"Validation mAP50 = "
        f"{current_map50:.4f}"
    )


    # --------------------------------------------------------
    # Global best 추적
    # --------------------------------------------------------

    if current_map50 > global_best_map50:

        global_best_map50 = current_map50
        global_best_stage = stage
        global_best_path = STAGE_BEST

        print(
            f">>> NEW GLOBAL BEST "
            f"(Stage {stage}, "
            f"mAP50={current_map50:.4f})"
        )


    # --------------------------------------------------------
    # 다음 stage는 last.pt에서 이어서 학습
    # --------------------------------------------------------

    current_model_path = STAGE_LAST


print()
print("=" * 60)
print("Training Finished")
print("=" * 60)

print(
    "Global Best Stage :",
    global_best_stage
)

print(
    "Global Best mAP50 :",
    f"{global_best_map50:.4f}"
)

print(
    "Global Best Path  :",
    global_best_path
)

print(
    "Final Last Path   :",
    current_model_path
)


Fine-tuning stage 1/5

Epoch 1
HRSID   : 3278
LS-SSDD : 1093
Total   : 4371
Ultralytics 8.4.135 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/HRSID_LSSDD_DYNAMIC/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=1e-05, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content

In [16]:
# ============================================================
# 16. Save Global Best + Final Last to Google Drive
# ============================================================

from pathlib import Path
import shutil


SAVE_DIR = Path(
    "/content/drive/MyDrive/"
    "SAR_AI_Ship_Detection/"
    "trained_models"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


DRIVE_BEST = (
    SAVE_DIR /
    "yolo26s_hrsid_lsssdd_dynamic3to1_5epoch_best.pt"
)

DRIVE_LAST = (
    SAVE_DIR /
    "yolo26s_hrsid_lsssdd_dynamic3to1_5epoch_last.pt"
)


# Global best
shutil.copy2(
    global_best_path,
    DRIVE_BEST
)


# Final stage last
shutil.copy2(
    current_model_path,
    DRIVE_LAST
)


print("Saved BEST:")
print(DRIVE_BEST)

print()

print("Saved LAST:")
print(DRIVE_LAST)

Saved BEST:
/content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models/yolo26s_hrsid_lsssdd_dynamic3to1_5epoch_best.pt

Saved LAST:
/content/drive/MyDrive/SAR_AI_Ship_Detection/trained_models/yolo26s_hrsid_lsssdd_dynamic3to1_5epoch_last.pt


In [17]:
# ============================================================
# 17. HRSID test YAML
# ============================================================

HRSID_TEST_YAML = (
    HRSID_YOLO /
    "data_test.yaml"
)


data = {
    "path": str(HRSID_YOLO),

    "train": "images/train",

    "val": "images/val",

    "test": "images/test",

    "names": {
        0: "ship"
    }
}


with open(
    HRSID_TEST_YAML,
    "w"
) as f:

    yaml.safe_dump(
        data,
        f,
        sort_keys=False
    )

In [ ]:
# ============================================================
# 18. Evaluate 3:1 Dynamic Mixed FT BEST on HRSID official test
# ============================================================

from ultralytics import YOLO


# ------------------------------------------------------------
# Load latest 3:1 / 5 epoch GLOBAL BEST model
# ------------------------------------------------------------

mixed_model = YOLO(
    str(DRIVE_BEST)
)


# ------------------------------------------------------------
# HRSID Official Test
# ------------------------------------------------------------

hrsid_metrics = mixed_model.val(

    data=str(HRSID_TEST_YAML),

    split="test",

    imgsz=800,

    batch=16,

    device=0,

    workers=2,

    plots=True,

    project="/content/runs",

    name="dynamic2to1_5epoch_best_HRSID_test"
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("========================================")
print("3:1 Dynamic Mixed FT BEST - HRSID TEST")
print("========================================")

print(
    f"Precision      : "
    f"{hrsid_metrics.box.mp:.4f}"
)

print(
    f"Recall         : "
    f"{hrsid_metrics.box.mr:.4f}"
)

print(
    f"mAP@0.5        : "
    f"{hrsid_metrics.box.map50:.4f}"
)

print(
    f"mAP@0.5:0.95   : "
    f"{hrsid_metrics.box.map:.4f}"
)

print("========================================")